# House Prices — 外れ値の決定と提出 / The Outlier Decision and a Submission

前作 [モデル選択の監査](https://www.kaggle.com/code/yosukeinada/house-prices-linear-model-choice-audit-jp-en)
は「LassoとElasticNetは実質同点。採用はまだしない」で終わった。
その後の fold 層別レビューで、この保留の裏に**もっと先に決めるべき問い**が
埋まっていたことが分かった。`Id=1299` — 学習データ最大の外れ値 — の扱いである。

このNotebookは、その決定を下すまでの過程を記録する。

1. 外れ値 2 行（`Id=524/1299`）の事実確認と、test に居る「双子」`Id=2550`
2. 扱いの選択肢 A〜E（5案）の公平な測定 — 失敗した測定過程も残す
3. 決定: **B（学習時のみ2行除外）** と、その代償として受けた賭け
4. モデル採用: **ElasticNet** と、B版提出ファイルの作成

> **English:** The previous audit ended with "no model adopted."
> A fold-stratified review then revealed a decision hiding underneath:
> what to do with `Id=1299`, the largest outlier in the training data —
> which has a near-twin (`Id=2550`) sitting in the test set.
> This notebook documents measuring five candidate treatments fairly
> (including the failed measurement attempts), the decision (drop 2 rows
> at fit time only), the model adoption (ElasticNet), and builds the
> submission.

**AI支援の開示 / AI assistance disclosure**:
測定コードの実装・実行と可視化はAIエージェント（Claude）が行い、
選択肢の決定（B）とモデル採用（ElasticNet）は人間が行った。
数値はすべてこのNotebook内または同一設計のローカル測定による実測値である。

In [1]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET, ID = "SalePrice", "Id"
OUTER_SEEDS, N_OUTER, N_INNER = [0, 4, 42], 5, 3

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

pandas: 2.3.3
scikit-learn: 1.7.2


In [2]:
def is_house_prices_pair(directory: Path) -> bool:
    train_path, test_path = directory / "train.csv", directory / "test.csv"
    if not train_path.exists() or not test_path.exists():
        return False
    train_columns = pd.read_csv(train_path, nrows=2).columns
    test_columns = pd.read_csv(test_path, nrows=2).columns
    return (TARGET in train_columns and TARGET not in test_columns
            and ID in train_columns and ID in test_columns)


KAGGLE_INPUT_ROOT = Path("/kaggle/input")
if KAGGLE_INPUT_ROOT.exists():
    candidates = sorted({p.parent for p in KAGGLE_INPUT_ROOT.rglob("train.csv")
                         if is_house_prices_pair(p.parent)})
    assert len(candidates) == 1, candidates
    DATA_DIR, OUTPUT_DIR, ENV = candidates[0], Path("/kaggle/working"), "Kaggle"
else:
    local = [p for p in (Path("data"), Path("../data"))
             if is_house_prices_pair(p)]
    assert len(local) == 1, "place train.csv/test.csv under ./data"
    DATA_DIR, OUTPUT_DIR, ENV = local[0], Path("submissions"), "Local"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
assert train.shape == (1460, 81) and test.shape == (1459, 80)
print(f"env={ENV}  train={train.shape}  test={test.shape}")

env=Local  train=(1460, 81)  test=(1459, 80)


## 1. なぜ外れ値の決定が先か / Why the outlier decision comes first

fold 層別のレビューで、モデルの順位が **`Id=1299` を含む3 fold とそれ以外で
逆転している**ことが分かった（難しい3 foldでは Ridge が最良、残り12 foldでは
ElasticNet が最良）。さらに測ってみると、外れ値の扱いを変えたときに CV が動く幅
（0.005246）は、正則化3種の間でモデルを変えたときの幅（0.001893）の**約2.8倍**
だった。

> 記録の訂正も残しておく: この比を最初「約10倍」と書いて公開情報にも伝播しかけた。
> 倍率の取り違えで、正しくは 2.77 倍である。効き幅は「まだ生きている選択肢がどれか」
> にも依存する（落選済みの LinearRegression を含めると 0.013474 に膨らみ、
> 決定の順位が逆転して見える）。

つまり「どのモデルか」より先に「`Id=1299` をどうするか」を決めないと、
モデル間に差があると言えるかどうかすら定まらない。

> **English:** Fold-stratified review showed the model ranking *reverses*
> on the 3 folds containing `Id=1299`. Changing the outlier treatment moves
> CV about 2.8x more than switching among the regularized models
> (initially misrecorded as "10x" — kept here as a correction record).
> The outlier decision therefore precedes model adoption.

## 2. 事実確認 / The facts about Id=524, 1299 — and their twin in the test set

In [3]:
cols = ["Id", "GrLivArea", "OverallQual", "YearBuilt", "Neighborhood",
        "SaleCondition", "SaleType", "SalePrice"]
big = train[train.GrLivArea > 4000][cols].copy()
big["usd_per_sf"] = (big.SalePrice / big.GrLivArea).round(1)
display(big.sort_values("GrLivArea", ascending=False))

# data_description.txt の定義:
#   Partial — Home was not completed when last assessed (associated with New Homes)
# 査定時点で家が完成していない。特徴量と価格が他の行と同じ前提で
# 対応していない可能性がある（定義から導ける仮説であり、機序の実証ではない）。

partial = train[train.SaleCondition == "Partial"].SalePrice
print(f"Partial {len(partial)}行の中央価格: {partial.median():,.0f}"
      f"（Normal中央 {train[train.SaleCondition=='Normal'].SalePrice.median():,.0f}）")
print("→ Partial自体は異常ではない。異常なのは「広大×Partial×激安」の2行だけ")

,Id,GrLivArea,OverallQual,YearBuilt,Neighborhood,SaleCondition,SaleType,SalePrice,usd_per_sf
1298,1299,5642,10,2008,Edwards,Partial,New,160000,28.4
523,524,4676,10,2007,Edwards,Partial,New,184750,39.5
1182,1183,4476,10,1996,NoRidge,Abnorml,WD,745000,166.4
691,692,4316,10,1994,NoRidge,Normal,WD,755000,174.9


Partial 125行の中央価格: 244,600（Normal中央 160,000）
→ Partial自体は異常ではない。異常なのは「広大×Partial×激安」の2行だけ


In [4]:
# 決定的な事実: test 側に同型の行が居る
twin = test[test.GrLivArea > 4000][["Id", "GrLivArea", "OverallQual",
                                    "YearBuilt", "Neighborhood",
                                    "SaleCondition"]]
display(twin)
# Id=2550: GrLivArea 5095・Edwards・Partial・2008 — Id=1299 のほぼ双子。
# 「外れ値を消せば精度が上がる」という一般論は、予測対象そのものが
# 外れ値と同型のとき、単純には成り立たない。

,Id,GrLivArea,OverallQual,YearBuilt,Neighborhood,SaleCondition
1089,2550,5095,10,2008,Edwards,Partial


## 3. 公平比較の設計 / A fair comparison design

行を除外して前後の CV を比べるのは**比較にならない**。一番難しい行を評価から
消せば CV は自動的に良くなるからである。そこで:

- 除外は**各外側 fold の学習部分にだけ**適用する
- 評価は常に元の validation 全行（`Id=1299` が validation に来たときは採点する）
- 特徴量 B2・seed 0/4/42 × 5-fold・内側3-fold・grid・Pipeline はすべて固定

この設計では、CV が既に「1299型を学習に持たないまま 1299 を予測する」状況を
含んでいる。これは test の `Id=2550` が置かれる状況と同型である。

> **English:** Dropping rows and comparing CV before/after is invalid —
> removing the hardest rows from *scoring* improves CV automatically.
> Here exclusions apply only to each outer fold's *training* portion;
> every variant is scored on identical validation rows. The CV then already
> contains the exact situation `Id=2550` poses at test time.

In [5]:
QUALITY = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
           "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond"]
QMAP = {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}


def b2(frame):
    out = frame.drop(columns=[TARGET, ID], errors="ignore").copy()
    for c in QUALITY:
        out[c] = out[c].map(QMAP).fillna(0).astype(float)
    return out


def pipe(features, model):
    num = features.select_dtypes(include="number").columns.tolist()
    cat = features.select_dtypes(exclude="number").columns.tolist()
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                              ("sc", StandardScaler())]), num),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="constant",
                                                    fill_value="Missing",
                                                    keep_empty_features=True)),
                              ("oh", OneHotEncoder(handle_unknown="ignore"))]),
             cat),
        ])),
        ("model", model),
    ])


X = b2(train)
y = np.log1p(train[TARGET])
EN_GRID = {"model__alpha": [0.0001, 0.0003, 0.001, 0.003, 0.01],
           "model__l1_ratio": [0.2, 0.5, 0.8, 0.9, 0.95, 1.0]}
HARD_FOLDS = {(0, 1), (4, 3), (42, 3)}   # Id=1299 が validation に来る 3 fold

DROP_B = set(train.loc[(train.GrLivArea > 4000)
                       & (train.SaleCondition == "Partial"), ID])
assert sorted(DROP_B) == [524, 1299]

In [6]:
# A（現状維持） vs B（学習時のみ2行除外）を ElasticNet で実測する。
# 15 外側 fold × 2 variant × GridSearch(30パラメータ×3内側fold)。数分かかる。
records = []
t0 = time.perf_counter()
for seed in OUTER_SEEDS:
    cv = KFold(n_splits=N_OUTER, shuffle=True, random_state=seed)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X), start=1):
        inner = KFold(n_splits=N_INNER, shuffle=True,
                      random_state=10_000 + seed * 10 + fold)
        va_ids = train.iloc[va_idx][ID].to_numpy()
        for variant, drop in [("A_keep_all", set()), ("B_drop2", DROP_B)]:
            keep = ~train.iloc[tr_idx][ID].isin(drop).to_numpy()
            search = GridSearchCV(
                pipe(X, ElasticNet(max_iter=200_000)), EN_GRID,
                scoring="neg_root_mean_squared_error",
                cv=inner, n_jobs=1, refit=True, error_score="raise")
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConvergenceWarning)
                search.fit(X.iloc[tr_idx][keep], y.iloc[tr_idx][keep])
            pred = search.predict(X.iloc[va_idx])
            records.append({
                "variant": variant, "outer_seed": seed, "outer_fold": fold,
                "stratum": "難fold3" if (seed, fold) in HARD_FOLDS else "残り12",
                "rmse": root_mean_squared_error(y.iloc[va_idx], pred),
                "rmse_excl_1299": root_mean_squared_error(
                    y.iloc[va_idx][va_ids != 1299], pred[va_ids != 1299])
                if (va_ids != 1299).sum() else np.nan,
            })
    print(f"seed {seed} done ({time.perf_counter()-t0:.0f}s)", flush=True)

fold_ab = pd.DataFrame(records)
summary = fold_ab.pivot_table(index="variant", values=["rmse", "rmse_excl_1299"],
                              aggfunc="mean")
strata = fold_ab.pivot_table(index=["stratum", "variant"], values="rmse",
                             aggfunc="mean")
display(summary.round(6))
display(strata.round(6))

wide = fold_ab.pivot_table(index=["outer_seed", "outer_fold"],
                           columns="variant", values="rmse")
diff = wide["B_drop2"] - wide["A_keep_all"]
print(f"ペア差 A→B: 平均 {diff.mean():+.6f} / 中央値 {diff.median():+.6f} / "
      f"B {(diff < 0).sum()}勝{(diff > 0).sum()}敗")

seed 0 done (42s)


seed 4 done (79s)


seed 42 done (119s)


,rmse,rmse_excl_1299
variant,,
A_keep_all,0.141726,0.126952
B_drop2,0.136481,0.120808


rmse
stratum variant             
残り12    A_keep_all  0.123154
        B_drop2     0.115680
難fold3  A_keep_all  0.216016
        B_drop2     0.219684

ペア差 A→B: 平均 -0.005246 / 中央値 -0.006613 / B 11勝2敗


### 読み取り / Reading

- B は A より平均で約 0.005 良く、15 fold 中 11 で勝つ。**平均と fold 多数決が
  一致**しており、特定 fold の偶然ではない。
- 改善の中身は「1299 が当たるようになった」ことではない。`Id=1299` 自身を採点から
  抜いた列（`rmse_excl_1299`）でも B は改善している。つまり **1299 に引きずられて
  いた係数が、他の全行に対して正常化した**。
- 代償は難fold3 の悪化（1299型を学習に持たないままの予測は当然悪くなる）。
  これは CV に計上済みで、その上で総合は B が勝つ。

> **English:** B beats A by ~0.005, winning 11/15 folds with mean and
> majority agreeing. The gain persists even when scoring excludes Id=1299
> itself — the improvement is coefficient normalization for *all other rows*,
> not fitting the outlier. The cost (worse hard folds) is already priced
> into the CV.

## 4. 5つの選択肢の全測定 / All five options, measured

A/B の実測は上のセルで行った。C/D/E は同一設計・同一 fold のローカル測定
（`13_outlier_options_de.py`〜`18_lasso_under_b.py`、リポジトリに保存）による。

| 選択肢 | 行の扱い | ElasticNet | Ridge | Huber |
|---|---|---:|---:|---:|
| A 現状維持 | 全行 | 0.141726 | 0.143619 | — |
| B 2行除外（GrLivArea>4000 かつ Partial） | 学習時のみ −2 | **0.136481** | 0.137311 | — |
| C 4行除外（GrLivArea>4000） | 学習時のみ −4 | 0.136686 | 0.137068 | — |
| D 頑健損失（Huber＋L2）で行は消さない | 全行 | — | — | 0.137249 |
| E Partial×面積の交互作用で行は消さない | 全行 | 0.142006 | 0.142252 | — |

**D の測定は一度失敗している**ことを残しておく。最初の grid（alpha ≤ 0.1）では
内側CVが両端に張り付いて 0.155705 という値が出た。端を追って3回 grid を広げた
結果、最適域は当初 grid の **3000倍以上外**（alpha ≈ 300）にあり、確定値 0.137249
は行を消さない B とほぼ同格だった。「頑健損失は効かない」と最初の測定だけで
結論していたら誤りだった。

**E は CV では効かないが、副作用が全選択肢で唯一逆向き**である（次節）。

> **English:** C/D/E were measured locally with the identical design and
> folds (scripts kept in the repo). D's first measurement was *wrong* —
> the inner CV pinned both grid edges; after widening the grid 3 times the
> optimum sat 3000x outside the initial range, and robust loss nearly
> matched B without deleting any rows. E does nothing for CV but is the
> only option whose side effect points the *other* way (next section).

## 5. CVに出ない副作用 / The side effect CV cannot see

学習時に 2 行を消すと、モデルは「広い＝高い」をより強く信じるようになる
（`GrLivArea` 係数が約3倍）。その結果、**test の双子 `Id=2550` への予測が
倍増する**。これは CV のどの数字にも現れない。実測して確かめる。

In [7]:
side = []
for variant, drop in [("A_keep_all", set()), ("B_drop2", DROP_B)]:
    keep = ~train[ID].isin(drop).to_numpy()
    search = GridSearchCV(pipe(X, ElasticNet(max_iter=200_000)), EN_GRID,
                          scoring="neg_root_mean_squared_error",
                          cv=KFold(N_INNER, shuffle=True, random_state=7),
                          n_jobs=1, refit=True, error_score="raise")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        search.fit(X[keep], y[keep])
    pred_test = np.expm1(search.predict(b2(test)))
    prep = search.best_estimator_.named_steps["prep"]
    names = list(prep.get_feature_names_out())
    coefs = np.asarray(
        search.best_estimator_.named_steps["model"].coef_).ravel()
    side.append({
        "variant": variant, "best": str(search.best_params_),
        "Id=2550": round(float(pred_test[test[ID] == 2550][0])),
        "test_max": round(float(pred_test.max())),
        "coef_GrLivArea": round(float(coefs[names.index("num__GrLivArea")]), 4),
    })
    if variant == "B_drop2":
        b_search = search          # 提出用に保持（採用構成そのもの）
        b_pred_test = pred_test
display(pd.DataFrame(side))
# 参考（同一設計のローカル測定）: D(Huber) 1,593,118 / E(交互作用) 874,510。
# E だけが予測を「下げる」方向 — 2行の割引パターンを係数として保持し
# 2550 に適用する唯一の選択肢だが、その学習根拠は train 内の2行しかない。

,variant,best,Id=2550,test_max,coef_GrLivArea
0,A_keep_all,"{'model__alpha': 0.0003, 'model__l1_ratio': 1.0}",1174562,1174562,0.0398
1,B_drop2,"{'model__alpha': 0.001, 'model__l1_ratio': 0.5}",1728747,1728747,0.1256


## 6. 決定 / The decision

**選択: B（学習時のみ2行除外）。** 判断の構造は3層ある。

1. **B > A / E はデータで決着済み** — 11勝2敗2分、平均と多数決が一致。
2. **B / C / D は CV で区別不能**（差 ≤ 0.0008、fold 標準偏差 ~0.04）。ここは
   方針で決めた: B は規則の狙いが最も正確で説明可能である
   （`data_description` の Partial 定義に根拠があり、正常な高額行を巻き込まない。
   `GrLivArea > 4000` の除去は Ames データセット原著 De Cock (2011) の推奨とも
   整合する）。D は行を消さない代わりにモデル族が Huber に固定され、
   「外れ値の扱い」と「モデル選択」という2つの決定の分離が壊れる。
3. **`Id=2550` の賭けを受けた。** B では 2550 の予測が 1.73M に倍増する。
   真値がどこにあっても B が総合で有利という試算（CV改善の test への転移を仮定）
   を信じる形である。この賭けの帰結は LB でしか測れない。

続いて**モデル採用**: B の下で3モデルを測ると ElasticNet 0.136481 /
Lasso 0.137267（EN に 4勝11敗）/ Ridge 0.137311。外れ値2行を除いても Lasso は
EN に追いつかず、順位反転はどの variant でも観測されなかった。**ElasticNet を
採用**した（線形モデル内の採用。非線形との比較は未実施のまま残る）。

> **English:** Decision: **B** — beats A/E decisively; within the
> statistical tie B/C/D, B has the most precise and explainable rule
> (consistent with De Cock's original guidance), and D would weld the
> outlier decision to a model-family change. The doubled `Id=2550`
> prediction is a bet accepted, measurable only on the leaderboard.
> Model adoption under B: **ElasticNet** (0.136481; Lasso loses 11/15
> paired folds even without the outliers). Non-linear models remain
> uncompared.

## 7. 提出ファイル / The submission

In [8]:
# 採用構成 = 上の B_drop2 の全学習 fit そのもの（b_search / b_pred_test）
submission = pd.DataFrame({ID: test[ID], TARGET: b_pred_test})

assert submission.shape == (1459, 2)
assert (submission[ID].to_numpy() == test[ID].to_numpy()).all()
assert submission[TARGET].notna().all()
assert np.isfinite(b_pred_test).all() and (b_pred_test > 0).all()

out_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(out_path, index=False)
print(f"-> {out_path}")
print(f"best params: {b_search.best_params_}")
print(f"Id=2550: {float(b_pred_test[test[ID] == 2550][0]):,.0f}")
print(f"range: {b_pred_test.min():,.2f} .. {b_pred_test.max():,.2f}")
submission.head()

-> submissions\submission.csv
best params: {'model__alpha': 0.001, 'model__l1_ratio': 0.5}
Id=2550: 1,728,747
range: 44,499.87 .. 1,728,747.24


,Id,SalePrice
0,1461,117704.949646
1,1462,152404.399895
2,1463,175651.746926
3,1464,199529.852688
4,1465,190961.150845


## 8. 限界と残り / Limits and what remains

- **`Id=2550` の賭けは未検証。** B の総合有利という試算は CV 改善が test へ
  転移するという仮定の上にある。実測で閉じる手段は A 版 / B 版の提出を
  リーダーボードで直接比べることだけである。
- **非線形モデルとの比較は未実施。** 今回の採用は線形モデル内の決定であり、
  勾配ブースティング等との比較は開いたまま。
- D（Huber）の確定測定は、grid の位置を外側 fold の結果を見ながら定めたため、
  grid 選定に外側情報が漏れている。探索段階の妥協として記録する。
- E の交互作用係数は train 内の2行だけから学習されており、係数の信頼性は低い。

> **English:** The `Id=2550` bet is unverified (only an A-vs-B leaderboard
> comparison can close it). Non-linear models remain uncompared. D's final
> grid placement leaked outer-fold information (an accepted exploration
> compromise), and E's interaction coefficient rests on just two training
> rows.